# WGCNA Analysis of Glycogene Expression in Liver Cancer Cell Lines (v2)

## Objective

Apply Weighted Gene Co-expression Network Analysis (WGCNA) to glycogene expression data 
extracted from the LINCS L1000 dataset in liver cancer cell lines (HEPG2)
to identify co-expression modules associated with drug response.

### Analysis Workflow
1. **Data Retrieval**: Retrieve expression data from Snowflake
2. **Preprocessing**: Verify z-score normalization, quality check
3. **Expression Matrix**: Create compound x gene matrix
4. **Filtering**: Remove low-variance genes (385 -> 289 genes)
5. **Correlation Calculation**: Pearson correlation between genes
6. **Soft Threshold Selection**: Scale-free topology fit (beta=4, R^2=0.80)
7. **TOM Calculation & Clustering**: Hierarchical clustering
8. **Dynamic Tree Cut**: Module detection (deepSplit=3, minClusterSize=5)
9. **Module Visualization**: Dendrogram, TOM heatmap
10. **Save Results**: Output module information
11. **Enrichment Analysis**: Pathway enrichment using GlycoEnzOnto

### Key Results
- **24 co-expression modules** identified
- Module 1 (37 genes): Glycan biosynthesis regulation (p=7.89x10^-8)
- Module 3 (31 genes): Lysosomal glycan degradation
- Module 4 (26 genes): Xenobiotic glucuronidation (p=4.48x10^-9)
- Module 6 (16 genes): Sugar nucleotide synthesis (p=5.04x10^-5)

## WGCNA Key Concepts Summary

### 1. Basic Concepts

| Term | Description |
|------|-------------|
| **TOM (Topological Overlap Matrix)** | Measures similarity in connection patterns between genes. Considers not only direct correlation but also shared "friends" |
| **Soft Threshold (beta)** | Raises correlation to a power for weighting. Achieves scale-free topology (few hub genes) |
| **Scale-free Topology Fit (R^2)** | Degree to which network has scale-free properties. R^2 > 0.8 is recommended |
| **Module** | Group of genes with similar co-expression patterns. Identified by hierarchical clustering |

### 2. Dynamic Tree Cut Parameters

**deep_split** (0-4): Controls sensitivity of module detection

| Value | Characteristics | Use Case |
|-------|-----------------|----------|
| 0-1 | Large modules | Broad biological processes |
| 2 (default) | Moderate | General analysis |
| **3 (this analysis)** | Finer granularity | Pathway-level analysis |
| 4 | Fine-grained modules | Specific functions |

**Reasons for choosing deep_split=3**:
- Focus on interpretation at GlycoEnzOnto pathway level
- Verified by biological interpretability of enrichment results
- Achieved appropriate granularity with 24 modules

### 3. Module Quality Metrics

| Metric | Meaning | Good Value |
|--------|---------|------------|
| **Silhouette Score** | Cluster separation. +1=perfect separation, 0=overlap, -1=misclassification | > 0.25 |
| **Intra-module Connectivity** | Average correlation among genes within module | Higher is better |
| **Inter-module Connectivity** | Average correlation between modules | Lower is better |
| **Modularity** | Difference between intra and inter | Higher is better |

### 4. Statistical Notes

**FDR Correction Scope**:
- X Per-module FDR: Correction only within each module -> Increased false positives
- **Global FDR**: All tests (module x pathway) corrected together

Example: 24 modules x 20 pathways = 480 tests
-> Benjamini-Hochberg correction on all 480 tests

### 5. Settings Used in This Analysis

```python
deep_split = 3  # Appropriate granularity for pathway-level analysis
```

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import os
import logging
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings('ignore')

# For Snowflake connection
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import serialization
from snowflake.connector import connect

# Configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

from utils.pathway_abbreviations import abbreviate_pathway

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams.update({
    'axes.labelsize': 18,
    'axes.titlesize': 16,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 16,
    'font.size': 16,
    'svg.fonttype': 'none',
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.linewidth': 1.2,
    'lines.linewidth': 1.5,
})

results_dir = project_root / 'notebooks' / 'results' / 'wgcna_v2'
results_dir.mkdir(parents=True, exist_ok=True)
print(f'Results directory: {results_dir}')

## 1. Snowflake Connection and Data Retrieval

In [ ]:
def load_private_key() -> bytes:
    """Load private key (DER format)"""
    key_path = os.path.expanduser('~/.ssh/snowflake_rsa_key.pem')
    with open(key_path, "rb") as key_file:
        private_key = serialization.load_pem_private_key(
            key_file.read(),
            password=None,
            backend=default_backend()
        )
    private_key_der = private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )
    return private_key_der


def connect_to_snowflake():
    """Connect to Snowflake"""
    try:
        conn = connect(
            user="KOREEDA",
            account="DUETMBM-LL33279",
            private_key=load_private_key(),
            warehouse="BIOINFORMATICS_XS",
            database="BIOINFORMATICS",
            schema="LINCS"
        )
        logger.info("Snowflake connection successful")
        return conn
    except Exception as e:
        logger.error(f"Snowflake connection error: {e}")
        raise


print('Connecting to Snowflake...')
conn = connect_to_snowflake()

In [ ]:
# Define liver cancer cell lines
liver_cells = ['HEPG2']

# Define metadata columns
metadata_columns = {
    'VALUE', 'canonical_smiles', 'cell', 'cmapid', 'compound_alias', 
    'dose', 'inchi_key', 'pertid', 'pertname', 'timepoint', 'sample_id'
}

# Get glycogene column names
column_query = """
SELECT COLUMN_NAME 
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE TABLE_SCHEMA = 'LINCS' 
AND TABLE_NAME = 'GLYCO_GENES_WIDE' 
ORDER BY COLUMN_NAME
"""

column_df = pd.read_sql(column_query, conn)
all_columns = column_df['COLUMN_NAME'].tolist()
glyco_genes = [col for col in all_columns if col not in metadata_columns]

print(f"Number of glycogenes: {len(glyco_genes)}")
print(f"Example genes: {', '.join(glyco_genes[:10])}")

## 2. Data Preprocessing and Quality Check

## 3. Create Compound x Gene Expression Matrix

In [ ]:
# Check gene expression data distribution
print('\n[Expression Data Distribution]')

# Statistics for each gene
gene_stats = []
for gene in glyco_genes:
    if gene in df_raw.columns:
        values = df_raw[gene].dropna()
        if len(values) > 0:
            gene_stats.append({
                'gene': gene,
                'mean': values.mean(),
                'std': values.std(),
                'min': values.min(),
                'max': values.max(),
                'zero_count': (values == 0).sum(),
                'zero_pct': (values == 0).sum() / len(values) * 100
            })

df_gene_stats = pd.DataFrame(gene_stats)

print(f"Expression value statistics:")
print(f"  Mean: {df_gene_stats['mean'].mean():.6f} ± {df_gene_stats['mean'].std():.6f}")
print(f"  Range: [{df_gene_stats['min'].min():.3f}, {df_gene_stats['max'].max():.3f}]")
print(f"  Zero value ratio: {df_gene_stats['zero_pct'].mean():.2f}%")

# Check low variance genes
low_variance = df_gene_stats[df_gene_stats['std'] < 0.001]
print(f"  Low variance genes(<0.001): {len(low_variance)}items")

## 4. Gene Filtering

In [ ]:
print('=' * 80)
print('[Creating Compound x Gene Matrix]')
print('=' * 80)

# Calculate median across all experimental conditions for each compound (robust to outliers)
print('Calculating median per compound...')
df_compound_avg = df_raw.groupby('pertname')[glyco_genes].median()

print(f"\nCompound x Gene matrix: {df_compound_avg.shape}")
print(f"Number of compounds: {len(df_compound_avg)}")
print(f"Number of genes: {len(glyco_genes)}")

# Missing value processing
missing_before = df_compound_avg.isna().sum().sum()
missing_pct = missing_before / df_compound_avg.size * 100
print(f"Missing values: {missing_before:,} / {df_compound_avg.size:,} ({missing_pct:.3f}%)")

# Replace missing values with 0 (treat as no expression change)
expression_matrix = df_compound_avg.fillna(0)
print(f"After missing value processing: {expression_matrix.isna().sum().sum()}  missing values")

# Basic statistics
print(f"\nMatrix statistics:")
print(f"  Median: {np.median(expression_matrix.values):.6f}")
print(f"  Standard deviation: {expression_matrix.values.std():.6f}")
print(f"  Range: [{expression_matrix.values.min():.3f}, {expression_matrix.values.max():.3f}]")

expression_matrix.head()

## 5. Calculate Gene-Gene Correlation Matrix

In [ ]:
print('=' * 80)
print('[Gene Filtering]')
print('=' * 80)

# 1. Variance-based filtering
gene_variances = expression_matrix.var()
variance_threshold = np.percentile(gene_variances, 25)  # Exclude bottom 25%

high_var_genes = gene_variances[gene_variances >= variance_threshold].index.tolist()
print(f"Variance filtering:")
print(f"  Threshold (25th percentile): {variance_threshold:.6f}")
print(f"  Retained genes: {len(high_var_genes)} / {len(glyco_genes)}")

# 2. Absolute value filtering (exclude extremely low expression changes)
mean_abs_expression = expression_matrix.abs().mean()
abs_threshold = np.percentile(mean_abs_expression, 10)  # Exclude bottom 10%

high_abs_genes = mean_abs_expression[mean_abs_expression >= abs_threshold].index.tolist()
print(f"\nAbsolute value filtering:")
print(f"  Threshold (10th percentile): {abs_threshold:.6f}")
print(f"  Retained genes: {len(high_abs_genes)} / {len(glyco_genes)}")

# Genes satisfying both conditions
filtered_genes = list(set(high_var_genes) & set(high_abs_genes))
filtered_genes.sort()

print(f"\nFinal filtering result:")
print(f"  Genes after filtering: {len(filtered_genes)}")
print(f"  Excluded genes: {len(glyco_genes) - len(filtered_genes)}")
print(f"  Retention rate: {len(filtered_genes) / len(glyco_genes) * 100:.1f}%")

# Matrix after filtering
expression_filtered = expression_matrix[filtered_genes]
print(f"\nMatrix after filtering: {expression_filtered.shape}")

# Examples of excluded genes
excluded_genes = [g for g in glyco_genes if g not in filtered_genes]
print(f"\nExcluded gene examples (first 10): {excluded_genes[:10]}")
print(f"Retained gene examples (first 10): {filtered_genes[:10]}")

## 6. Soft Thresholding Power Selection

In [ ]:
print('=' * 80)
print('[Gene-Gene Correlation Matrix Calculation]')
print('=' * 80)

# [Fix] Calculate gene-gene correlation correctly
# expression_filtered: (compounds x genes) = (5057 x 289)
# .corr() calculates correlation between columns (genes) -> (genes x genes) = (289 x 289)
print(f"Expression matrix: {expression_filtered.shape} (compounds x genes)")

# Calculate Pearson correlation matrix (no transpose needed!)
print('Calculating gene-gene correlation...')
gene_corr = expression_filtered.corr(method='pearson')
print(f"Correlation matrix: {gene_corr.shape} (genes x genes)")

# Correlation matrix statistics
# Upper triangle only (excluding self-correlation)
mask = np.triu(np.ones_like(gene_corr), k=1).astype(bool)
corr_values = gene_corr.values[mask]

print(f"\nCorrelation statistics:")
print(f"  Number of pairs: {len(corr_values):,}")
print(f"  Mean: {corr_values.mean():.6f}")
print(f"  Standard deviation: {corr_values.std():.6f}")
print(f"  Range: [{corr_values.min():.3f}, {corr_values.max():.3f}]")
print(f"  25th percentile: {np.percentile(corr_values, 25):.3f}")
print(f"  75th percentile: {np.percentile(corr_values, 75):.3f}")

# Proportion of strong correlations
strong_pos = (corr_values > 0.3).sum()
strong_neg = (corr_values < -0.3).sum()
moderate = (np.abs(corr_values) > 0.1).sum()

print(f"\nCorrelation strength distribution:")
print(f"  Strong positive correlation (>0.3): {strong_pos:,} ({strong_pos/len(corr_values)*100:.2f}%)")
print(f"  Strong negative correlation (<-0.3): {strong_neg:,} ({strong_neg/len(corr_values)*100:.2f}%)")
print(f"  Moderate or higher (|r|>0.1): {moderate:,} ({moderate/len(corr_values)*100:.2f}%)")

# Correlation visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(corr_values, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(corr_values.mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Mean={corr_values.mean():.3f}')
axes[0].set_xlabel('Correlation Coefficient')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Gene-Gene Correlations')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Cumulative distribution
sorted_corr = np.sort(corr_values)
cumulative = np.arange(1, len(sorted_corr) + 1) / len(sorted_corr)
axes[1].plot(sorted_corr, cumulative, color='coral', linewidth=2)
axes[1].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Correlation Coefficient')
axes[1].set_ylabel('Cumulative Probability')
axes[1].set_title('Cumulative Distribution')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(results_dir / 'gene_correlation_distribution.png', dpi=300, bbox_inches='tight')
plt.savefig(results_dir / 'gene_correlation_distribution.svg', format='svg', bbox_inches='tight')
plt.show()

print(f"\nCorrelation matrix calculation complete")

## 7. Adjacency Matrix, TOM Calculation and Hierarchical Clustering

Quantify co-expression relationships between genes and perform hierarchical clustering.
Module partitioning is done using Dynamic Tree Cut.

In [ ]:
# Soft threshold results visualization and power selection
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# R-squared plot
ax = axes[0, 0]
ax.plot(df_sft['power'], df_sft['rsquared'], 'o-', color='steelblue', linewidth=2, markersize=6)
ax.axhline(0.8, color='red', linestyle='--', label='R²=0.8 threshold')
ax.axhline(0.5, color='orange', linestyle='--', label='R²=0.5 threshold')
ax.set_xlabel('Soft Threshold Power (β)')
ax.set_ylabel('Scale-free R²')
ax.set_title('Scale-free Topology Fit')
ax.legend()
ax.grid(alpha=0.3)

# Mean connectivity plot
ax = axes[0, 1]
ax.plot(df_sft['power'], df_sft['mean_k'], 'o-', color='coral', linewidth=2, markersize=6)
ax.set_xlabel('Soft Threshold Power (β)')
ax.set_ylabel('Mean Connectivity')
ax.set_title('Mean Connectivity')
ax.grid(alpha=0.3)

# log(k) vs log(p(k)) examples (for several powers)
ax = axes[1, 0]
example_powers = [1, 5, 10, 15]
colors = ['red', 'blue', 'green', 'purple']

for i, power in enumerate(example_powers):
    adj = np.abs(gene_corr.values) ** power
    k_values = adj.sum(axis=1) - 1
    k_values = k_values[k_values > 0]
    
    if len(k_values) > 5:
        hist, bins = np.histogram(k_values, bins=min(30, len(k_values)//2))
        bin_centers = (bins[:-1] + bins[1:]) / 2
        
        positive_mask = (hist > 0) & (bin_centers > 0)
        if positive_mask.sum() > 3:
            x_log = np.log10(bin_centers[positive_mask])
            y_log = np.log10(hist[positive_mask])
            
            ax.scatter(x_log, y_log, color=colors[i], alpha=0.6, s=30, label=f'β={power}')

ax.set_xlabel(r'$\log_{10}(k)$')
ax.set_ylabel(r'$\log_{10}(p(k))$')
ax.set_title('Scale-free Topology Check')
ax.legend()
ax.grid(alpha=0.3)

# Power selection statistics table
ax = axes[1, 1]
ax.axis('off')

# Power selection criteria
# 1. Minimum power achieving R² >= 0.8
# 2. If not available, minimum power achieving R² >= 0.5
# 3. If not available, maximum power maintaining mean connectivity > 50

high_rsq = df_sft[df_sft['rsquared'] >= 0.8]
med_rsq = df_sft[df_sft['rsquared'] >= 0.5]
good_conn = df_sft[df_sft['mean_k'] >= 50]

if len(high_rsq) > 0:
    selected_power = high_rsq.iloc[0]['power']
    selection_reason = f"R² >= 0.8 achieved"
elif len(med_rsq) > 0:
    selected_power = med_rsq.iloc[0]['power']
    selection_reason = f"R² >= 0.5 achieved"
elif len(good_conn) > 0:
    selected_power = good_conn.iloc[-1]['power']  # Maximum power
    selection_reason = f"Mean connectivity >= 50"
else:
    # Compromise: select moderate power
    idx = len(df_sft) // 2
    selected_power = df_sft.iloc[idx]['power']
    selection_reason = f"Moderate power (compromise)"

selected_row = df_sft[df_sft['power'] == selected_power].iloc[0]

info_text = f"""Selected power: β = {selected_power}
Selection reason: {selection_reason}

Metrics at selected power:
  R² = {selected_row['rsquared']:.3f}
  Mean K = {selected_row['mean_k']:.1f}
  Median K = {selected_row['median_k']:.1f}
  Max K = {selected_row['max_k']:.1f}

Criteria:
  Ideal: R² >= 0.8
  Acceptable: R² >= 0.5
  Minimum: Mean K >= 50"""

ax.text(0.05, 0.95, info_text, transform=ax.transAxes,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

plt.tight_layout()
plt.savefig(results_dir / 'soft_threshold_selection.png', dpi=300, bbox_inches='tight')
plt.savefig(results_dir / 'soft_threshold_selection.svg', format='svg', bbox_inches='tight')
plt.show()

print(f'\n【Selected power】: β = {selected_power}')
print(f'[Reason]: {selection_reason}')
print(f'【R²】: {selected_row["rsquared"]:.3f}')
print(f'【Mean Connectivity】: {selected_row["mean_k"]:.1f}')

# === Sensitivity analysis: Force selection of β=3 ===
selected_power = 4
selected_row = df_sft[df_sft["power"] == selected_power].iloc[0]
selection_reason = "Sensitivity analysis: β=3 (prioritizing higher connectivity)"
print(f"\n=== Sensitivity Analysis Mode ===")
print(f"β = {selected_power} changed to")
print(f"R² = {selected_row['rsquared']:.3f}")
print(f"Mean Connectivity = {selected_row['mean_k']:.1f}")

## 8. Module Detection with Dynamic Tree Cut

Dynamic Tree Cut is a method that adaptively detects clusters from dendrogram shape.
Unlike traditional methods using fixed cut height, it automatically identifies modules based on data.

**Parameters**:
- `deepSplit=3`: Appropriate granularity for pathway-level analysis
- `minClusterSize=5`: Minimum module size

In [ ]:
print('=' * 80)
print('[Distance Matrix Calculation and Hierarchical Clustering]')
print('=' * 80)

# Calculate distance matrix (1 - TOM)
print('Calculating distance matrix...')
distance_matrix = 1 - TOM
np.fill_diagonal(distance_matrix, 0)  # Diagonal elements are 0

print(f'Distance matrix: {distance_matrix.shape}')

# Distance matrix statistics
dist_values = distance_matrix[np.triu_indices_from(distance_matrix, k=1)]
print(f'Distance statistics:')
print(f'  Mean: {dist_values.mean():.6f}')
print(f'  Standard deviation: {dist_values.std():.6f}')
print(f'  Range: [{dist_values.min():.6f}, {dist_values.max():.6f}]')

# Hierarchical clustering
print('\nRunning hierarchical clustering...')
from scipy.spatial.distance import squareform

# Convert to condensed distance matrix
distance_condensed = squareform(distance_matrix, checks=False)
print(f'CondensedDistance matrix: {distance_condensed.shape}')

# Clustering using Ward method
linkage_matrix = linkage(distance_condensed, method='ward')
print(f'Linkage matrix: {linkage_matrix.shape}')

print('Hierarchical clustering complete')

In [ ]:
import os
os.environ['R_HOME'] = '/Library/Frameworks/R.framework/Versions/4.4-x86_64/Resources'

# WGCNA Module Detection (R version)
print('=' * 80)
print('[WGCNA Module Detection (R version)]')
print('=' * 80)

import rpy2.robjects as ro
from rpy2.robjects import numpy2ri, default_converter
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter

# Import R packages
WGCNA = importr('WGCNA')
dynamicTreeCut = importr('dynamicTreeCut')

# numpy-R conversion context
np_cv = default_converter + numpy2ri.converter

# Filter expression data by filtered_genes
print(f'Original expression_matrix shape: {expression_matrix.shape}')
print(f'filtered_genes: {len(filtered_genes)}')

expr_filtered = expression_matrix[filtered_genes].values
print(f'After filtering: {expr_filtered.shape}')

with localconverter(np_cv):
    # WGCNA expects rows=samples, columns=genes
    expr_r = ro.r.matrix(ro.FloatVector(expr_filtered.flatten()), 
                         nrow=expr_filtered.shape[0], byrow=True)
    ro.r.assign('datExpr', expr_r)
    ro.r.assign('geneNames', ro.StrVector(filtered_genes))
    ro.r('colnames(datExpr) <- geneNames')
    
    # Soft threshold power
    ro.r.assign('power', selected_power)
    
    # TOM calculation (R WGCNA)
    print(f'\nCalculating TOM (R WGCNA, power={selected_power}）...')
    ro.r('TOM <- TOMsimilarityFromExpr(datExpr, power = power, TOMType = "unsigned", verbose = 0)')
    ro.r('dissTOM <- 1 - TOM')
    
    # Hierarchical clustering
    print('Running hierarchical clustering...')
    ro.r('geneTree <- hclust(as.dist(dissTOM), method = "average")')
    
    # Dynamic Tree Cut
    print('Running Dynamic Tree Cut...')
    print('  deepSplit: 3, minClusterSize: 3')
    ro.r('dynamicMods <- cutreeDynamic(dendro = geneTree, distM = dissTOM, deepSplit = 2, pamRespectsDendro = FALSE, minClusterSize = 3)')
    
    # Module colors
    ro.r('dynamicColors <- labels2colors(dynamicMods)')
    
    # Get results to Python
    dynamicMods = ro.r('dynamicMods')
    dynamicColors = ro.r('dynamicColors')
    module_labels = np.array([int(x) for x in dynamicMods], dtype=np.int32)
    module_colors = list(dynamicColors)
    
    # Get TOM to Python
    TOM_r = ro.r('TOM')
    TOM = np.array(TOM_r).reshape((len(filtered_genes), len(filtered_genes)))

# Display statistics
n_unassigned = int((module_labels == 0).sum())
n_modules = len(set(module_labels)) - (1 if 0 in module_labels else 0)

print(f'\n[Results]')
print(f'  Number of modules: {n_modules}')
print(f'  Unassigned genes: {n_unassigned}')

module_sizes = pd.Series(module_labels[module_labels > 0]).value_counts().sort_index()
print(f'  Size range: {module_sizes.min()} - {module_sizes.max()}')
print(f'  Mean size: {module_sizes.mean():.1f}')

# Create df_modules
df_modules = pd.DataFrame({
    'gene': filtered_genes,
    'module': module_labels,
    'color': module_colors
})

print(f'\nAssigned genes: {len(df_modules[df_modules["module"] > 0])} / {len(df_modules)}')
print('=' * 80)

# Save: gene_modules.csv
df_modules_to_save = df_modules[df_modules['module'] > 0][['gene', 'module']].copy()
df_modules_to_save.to_csv(results_dir / 'gene_modules.csv', index=False)
print(f'\nSaved: {results_dir / "gene_modules.csv"} ({len(df_modules_to_save)} genes)')


## 9. Module Visualization

Visualize the structure of identified modules.

In [ ]:
# Draw dendrogram (R)
print('=' * 80)
print('[Dendrogram Drawing (R WGCNA)]')
print('=' * 80)

# Draw dendrogram in R
ro.r('''
# Output settings
pdf_file <- file.path("''' + str(results_dir) + '''", "wgcna_dendrogram_modules.pdf")
svg_file <- file.path("''' + str(results_dir) + '''", "wgcna_dendrogram_modules.svg")
png_file <- file.path("''' + str(results_dir) + '''", "wgcna_dendrogram_modules.png")

# PDF output
pdf(pdf_file, width = 14, height = 8)
plotDendroAndColors(geneTree, dynamicColors,
                    "Module colors",
                    dendroLabels = FALSE,
                    hang = 0.03,
                    addGuide = TRUE,
                    guideHang = 0.05,
                    main = "Gene Dendrogram with Module Colors (Dynamic Tree Cut)")
dev.off()

# SVG output
svg(svg_file, width = 14, height = 8)
plotDendroAndColors(geneTree, dynamicColors,
                    "Module colors",
                    dendroLabels = FALSE,
                    hang = 0.03,
                    addGuide = TRUE,
                    guideHang = 0.05,
                    main = "Gene Dendrogram with Module Colors (Dynamic Tree Cut)")
dev.off()

# PNG output
png(png_file, width = 1400, height = 800, res = 100)
plotDendroAndColors(geneTree, dynamicColors,
                    "Module colors",
                    dendroLabels = FALSE,
                    hang = 0.03,
                    addGuide = TRUE,
                    guideHang = 0.05,
                    main = "Gene Dendrogram with Module Colors (Dynamic Tree Cut)")
dev.off()

cat("Dendrogram saved to:", pdf_file, "\n")
''')

# For display in Jupyter
from IPython.display import Image, display
import os

png_path = results_dir / 'wgcna_dendrogram_modules.png'
if os.path.exists(png_path):
    display(Image(filename=str(png_path)))
    print(f'Dendrogram saved: {results_dir}')
else:
    print('Failed to display image')


## 11. Enrichment Analysis per Module

Analyze which GlycoEnzOnto pathways are enriched in each module.

In [ ]:
# Load GlycoEnzOnto GMT file
print('=' * 80)
print('[Loading GlycoEnzOnto]')
print('=' * 80)

gmt_file = project_root / 'GlycoEnzOnto' / 'GlycoEnzOnto.gmt'

# Verify file exists
if not gmt_file.exists():
    raise FileNotFoundError(f'GMT file not found: {gmt_file}')

print(f'GMT file: {gmt_file}')

def read_gmt(gmt_path):
    """Read GMT file and return pathway dictionary"""
    pathways = {}
    with open(gmt_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            pathway_name = parts[0]
            # Remove quotes
            pathway_name = pathway_name.strip('"').strip("'")
            genes = parts[2:]  # First 2 columns are pathway name and description
            # Also remove quotes from gene names
            genes = [g.strip('"').strip("'") for g in genes]
            pathways[pathway_name] = set(genes)
    return pathways

pathways = read_gmt(gmt_file)

print(f'\nGlycoEnzOnto pathway information:')
print(f'  Total pathways: {len(pathways)}')
print(f'  Pathway examples:')
for i, (name, genes) in enumerate(list(pathways.items())[:5], 1):
    print(f'    {i}. {name}: {len(genes)}genes')

print('\nGlycoEnzOnto loading complete')

In [ ]:
# Define enrichment analysis function (with condition checks)
from scipy.stats import fisher_exact

def perform_enrichment_analysis(module_genes, all_genes, pathways, min_overlap=2, verbose=False):
    """
    Enrichment analysis of module genes
    
    Parameters:
    -----------
    module_genes : set
        Gene set contained in module
    all_genes : set
        All gene set (background)
    pathways : dict
        Dictionary of pathway name -> gene set
    min_overlap : int
        Minimum overlap count (default: 2)
    verbose : bool
        Display detailed information
    
    Returns:
    --------
    pd.DataFrame
        Enrichment results
    """
    results = []
    
    n_module = len(module_genes)
    n_background = len(all_genes)
    
    # Counters for statistical checks
    total_tests = 0
    small_expected_count = 0  # Cases with expected value < 5
    
    for pathway_name, pathway_genes in pathways.items():
        # Intersection of pathway genes and all genes
        pathway_genes_in_background = pathway_genes & all_genes
        
        if len(pathway_genes_in_background) == 0:
            continue
        
        # Overlap
        overlap = module_genes & pathway_genes_in_background
        n_overlap = len(overlap)
        
        if n_overlap < min_overlap:
            continue
        
        # 2x2 contingency table
        a = n_overlap  # In module & in pathway
        b = len(pathway_genes_in_background) - a  # Outside module & in pathway
        c = n_module - a  # In module & outside pathway
        d = n_background - n_module - b  # Outside module & outside pathway
        
        # Expected value check (Fisher's exact test can be used with expected < 5, but recorded)
        expected_a = (n_module * len(pathway_genes_in_background)) / n_background
        if expected_a < 5:
            small_expected_count += 1
        
        # Negative cell check
        if d < 0:
            if verbose:
                print(f"WARNING: negative cell count in {pathway_name}")
            continue
        
        total_tests += 1
        
        # Fisher's exact test
        oddsratio, pvalue = fisher_exact([[a, b], [c, d]], alternative='greater')
        
        # Enrichment ratio
        fold_enrichment = n_overlap / expected_a if expected_a > 0 else 0
        
        results.append({
            'pathway': pathway_name,
            'overlap': n_overlap,
            'module_size': n_module,
            'pathway_size': len(pathway_genes_in_background),
            'expected': expected_a,
            'fold_enrichment': fold_enrichment,
            'odds_ratio': oddsratio,
            'p_value': pvalue,
            'genes': ','.join(sorted(overlap))
        })
    
    if not results:
        return pd.DataFrame(), {'total_tests': 0, 'small_expected': 0}
    
    df_results = pd.DataFrame(results)
    
    # Statistical information
    stats_info = {
        'total_tests': total_tests,
        'small_expected': small_expected_count,
        'small_expected_pct': 100 * small_expected_count / total_tests if total_tests > 0 else 0
    }
    
    # Sort by p-value
    df_results = df_results.sort_values('p_value')
    
    return df_results, stats_info

print('Enrichment analysis function defined (with condition checks)')


In [ ]:
# Enrichment analysis (Global FDR correction)
print('=' * 80)
print('[Enrichment Analysis]')
print('=' * 80)

from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

# GlycoEnzOnto Load GMT file
gmt_file = project_root / 'GlycoEnzOnto' / 'GlycoEnzOnto.gmt'

def read_gmt(gmt_path):
    pathways = {}
    with open(gmt_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            pathway_name = parts[0].strip('"').strip("'")
            genes = [g.strip('"').strip("'") for g in parts[2:]]
            pathways[pathway_name] = set(genes)
    return pathways

pathways = read_gmt(gmt_file)
all_genes_set = set(filtered_genes)

print(f'Number of pathways: {len(pathways)}')
print(f'Background gene count: {len(all_genes_set)}')

# Counters for statistical checks
total_tests = 0
small_expected_count = 0  # Cases with expected value < 5

# Enrichment analysis
enrichment_results = []

for module_id in sorted(df_modules['module'].unique()):
    if module_id == 0:  # Skip unassigned module
        continue
        
    module_genes = set(df_modules[df_modules['module'] == module_id]['gene'])
    n_module = len(module_genes)
    
    for pathway_name, pathway_genes in pathways.items():
        pathway_in_background = pathway_genes & all_genes_set
        if len(pathway_in_background) < 2:
            continue
        
        overlap = module_genes & pathway_in_background
        n_overlap = len(overlap)
        
        if n_overlap < 2:
            continue
        
        # 2x2 contingency table
        a = n_overlap
        b = len(pathway_in_background) - a
        c = n_module - a
        d = len(all_genes_set) - n_module - b
        
        # Expected value check
        expected = (n_module * len(pathway_in_background)) / len(all_genes_set)
        if expected < 5:
            small_expected_count += 1
        
        # Negative cell check
        if d < 0:
            print(f"WARNING: negative cell in module {module_id}, {pathway_name}")
            continue
        
        total_tests += 1
        
        # Fisher test
        oddsratio, pvalue = fisher_exact([[a, b], [c, d]], alternative='greater')
        fold_enrichment = n_overlap / expected if expected > 0 else 0
        
        enrichment_results.append({
            'module': module_id,
            'pathway': pathway_name,
            'overlap': n_overlap,
            'module_size': n_module,
            'pathway_size': len(pathway_in_background),
            'expected': expected,
            'fold_enrichment': fold_enrichment,
            'odds_ratio': oddsratio,
            'p_value': pvalue,
            'genes': ','.join(sorted(overlap))
        })

df_enrichment = pd.DataFrame(enrichment_results)

# Display statistical information
print()
print('[Statistical Information]')
print(f'  Total tests: {total_tests}')
print(f'  Cases with expected value < 5: {small_expected_count} ({100*small_expected_count/total_tests:.1f}%)')
print(f'  → Fisher\'s exact testis valid even with expected < 5 (unlike chi-square test)')

# Global FDR correction (all modules x all pathways corrected together)
print()
print('[FDR Correction]')
print('  Method: Benjamini-Hochberg (Global FDR)')
print('  Scope: All tests corrected together (not per-module)')

if len(df_enrichment) > 0:
    _, qvalues, _, _ = multipletests(df_enrichment['p_value'], method='fdr_bh')
    df_enrichment['q_value'] = qvalues
    
    # Report results at multiple FDR thresholds
    print()
    print('[Number of Significant Enrichments]')
    for fdr_th in [0.01, 0.05, 0.10, 0.20]:
        n_sig = (df_enrichment['q_value'] < fdr_th).sum()
        print(f'  FDR < {fdr_th}: {n_sig}entries')
    
    # Main threshold
    df_sig = df_enrichment[df_enrichment['q_value'] < 0.05].copy()
    
    # Effect size filter (optional)
    df_sig_effect = df_enrichment[
        (df_enrichment['q_value'] < 0.05) & 
        (df_enrichment['fold_enrichment'] > 1.5)
    ].copy()
    print(f'  FDR < 0.05 & Fold > 1.5: {len(df_sig_effect)}entries')
    
    # Number of significant modules
    sig_modules = df_sig['module'].nunique()
    total_modules = df_modules[df_modules['module'] > 0]['module'].nunique()
    print(f'\nModules with significant enrichment: {sig_modules}/{total_modules}')
    
    # Saved
    df_enrichment.to_csv(results_dir / 'module_pathway_enrichments.csv', index=False)
    print(f'\nSaved: {results_dir}/module_pathway_enrichments.csv')

print('=' * 80)


In [ ]:
# Module dot plot (publication quality)
print('=' * 80)
print('[Enrichment Dot Plot]')
print('=' * 80)

# Extract enrichments with FDR < 0.05
df_plot = df_enrichment[df_enrichment['q_value'] < 0.05].copy()

if len(df_plot) > 0:
    # Create pathway display name
    df_plot['pathway_display'] = df_plot['pathway'].str.replace('_', ' ').str.title()
    df_plot['pathway_display'] = df_plot['pathway_display'].str.replace(' Pathway', '')
    df_plot['pathway_display'] = df_plot['pathway_display'].apply(abbreviate_pathway)
    
    # Calculate -log10(FDR)
    df_plot['neg_log10_fdr'] = -np.log10(df_plot['q_value'].clip(lower=1e-10))
    
    # Sort by module ID
    module_ids_sorted = sorted(df_plot['module'].unique())
    
    # Sort pathways by frequency
    pathway_counts = df_plot.groupby('pathway_display').size().sort_values(ascending=False)
    pathway_order = pathway_counts.index.tolist()
    
    # Figure size - compact x-axis
    n_modules_plot = df_plot['module'].nunique()
    fig_width = max(10, n_modules_plot * 0.4 + 12)
    fig_height = max(8, len(pathway_order) * 0.5 + 2)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    
    # Size and color range
    min_count = df_plot['overlap'].min()
    max_count = df_plot['overlap'].max()
    vmin = df_plot['neg_log10_fdr'].min()
    vmax = df_plot['neg_log10_fdr'].max()
    
    # Dummy scatter for colorbar
    scatter_dummy = ax.scatter([], [], c=[], cmap='Reds', vmin=vmin, vmax=vmax)
    
    # Module position mapping
    module_to_x = {m: i for i, m in enumerate(module_ids_sorted)}
    
    # Plot each enrichment
    for _, row in df_plot.iterrows():
        x = module_to_x[row['module']]
        y = pathway_order.index(row['pathway_display'])
        
        # Size (gene count)
        if max_count > min_count:
            size = 300 + 900 * (row['overlap'] - min_count) / (max_count - min_count)
        else:
            size = 600
        
        ax.scatter(x, y, s=size, c=[row['neg_log10_fdr']], 
                  cmap='Reds', vmin=vmin, vmax=vmax,
                  edgecolors='black', linewidth=0.8, alpha=0.85)
    
    # Axis settings
    ax.set_xticks(range(len(module_ids_sorted)))
    ax.set_yticks(range(len(pathway_order)))
    ax.set_xticklabels([f'M{m}' for m in module_ids_sorted], rotation=45, ha='right')
    ax.set_yticklabels(pathway_order, fontsize=14)
    
    ax.set_xlim(-0.5, len(module_ids_sorted) - 0.5)
    ax.set_ylim(-0.5, len(pathway_order) - 0.5)
    
    ax.set_xlabel('Module', fontweight='bold')
    ax.set_ylabel('Pathway', fontweight='bold')
    ax.set_title(f'Pathway Enrichment ({n_modules_plot} modules, FDR < 0.05)', fontweight='bold')
    
    ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
    
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.2)
    
    # Colorbar
    cbar = plt.colorbar(scatter_dummy, ax=ax, fraction=0.015, pad=0.02, shrink=0.5)
    cbar.set_label(r'$-\log_{10}(\text{FDR})$', labelpad=10)
    cbar.ax.tick_params(labelsize=14)
    cbar.outline.set_linewidth(1.2)
    
    # Size legend
    legend_counts = [int(min_count), int((min_count + max_count) / 2), int(max_count)]
    legend_handles = []
    
    for count in legend_counts:
        if max_count > min_count:
            size = 300 + 900 * (count - min_count) / (max_count - min_count)
        else:
            size = 600
        legend_handles.append(
            plt.scatter([], [], s=size, c='gray', alpha=0.7, 
                       edgecolors='black', linewidth=0.8)
        )
    
    legend = ax.legend(
        legend_handles,
        [f'{c} genes' for c in legend_counts],
        title='Gene Count',
        loc='upper left',
        bbox_to_anchor=(1.02, 1.0),
        framealpha=0.95,
        edgecolor='black',
        fancybox=False,
        labelspacing=3.0,
        borderpad=1.0
    )
    legend.get_frame().set_linewidth(1.2)
    
    plt.tight_layout()
    plt.savefig(results_dir / 'dtc_module_enrichment_dotplot.png', dpi=600, bbox_inches='tight')
    plt.savefig(results_dir / 'dtc_module_enrichment_dotplot.svg', format='svg', bbox_inches='tight')
    plt.show()
    
    print(f'Saved: {results_dir}/dtc_module_enrichment_dotplot.png')
    print(f'Number of significant enrichments: {len(df_plot)}entries')
    print(f'Number of modules included: {n_modules_plot}')
else:
    print('No significant enrichments (FDR < 0.05)')

print('=' * 80)

In [ ]:
# Nominal p < 0.05 dot plot (covering all modules)
print('=' * 80)
print('[Enrichment Dot Plot (nominal p < 0.05)]')
print('=' * 80)

# --- Data preparation ---
# Entries with nominal p < 0.05
df_nom = df_enrichment[df_enrichment['p_value'] < 0.05].copy()
df_nom['significance'] = 'nominal'
# Overwrite labels for FDR < 0.05
df_nom.loc[df_nom['q_value'] < 0.05, 'significance'] = 'FDR'

# Identify modules not meeting nominal p < 0.05 -> suggestive (top-1 pathway)
all_modules = sorted(df_enrichment['module'].unique())
covered_modules = set(df_nom['module'].unique())
missing_modules = [m for m in all_modules if m not in covered_modules]

if missing_modules:
    suggestive_rows = []
    for mod in missing_modules:
        df_mod = df_enrichment[df_enrichment['module'] == mod]
        if len(df_mod) > 0:
            best = df_mod.loc[df_mod['p_value'].idxmin()]
            row = best.to_dict()
            row['significance'] = 'suggestive'
            suggestive_rows.append(row)
    if suggestive_rows:
        df_nom = pd.concat([df_nom, pd.DataFrame(suggestive_rows)], ignore_index=True)

print(f'  FDR < 0.05: {(df_nom["significance"] == "FDR").sum()} entries')
print(f'  nominal p < 0.05: {(df_nom["significance"] == "nominal").sum()} entries')
print(f'  suggestive (top-1): {(df_nom["significance"] == "suggestive").sum()} entries')
print(f'  Number of modules: {df_nom["module"].nunique()} / {len(all_modules)}')

# --- Plot ---
df_plot2 = df_nom.copy()
df_plot2['pathway_display'] = df_plot2['pathway'].str.replace('_', ' ').str.title()
df_plot2['pathway_display'] = df_plot2['pathway_display'].str.replace(' Pathway', '')
df_plot2['pathway_display'] = df_plot2['pathway_display'].apply(abbreviate_pathway)
df_plot2['neg_log10_p'] = -np.log10(df_plot2['p_value'].clip(lower=1e-10))

module_ids_sorted = sorted(df_plot2['module'].unique())
pathway_counts = df_plot2.groupby('pathway_display').size().sort_values(ascending=False)
pathway_order = pathway_counts.index.tolist()

n_modules_plot = len(module_ids_sorted)
fig_width = max(10, n_modules_plot * 0.4 + 12)
fig_height = max(8, len(pathway_order) * 0.5 + 2)
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

min_count = df_plot2['overlap'].min()
max_count = df_plot2['overlap'].max()
vmin = df_plot2['neg_log10_p'].min()
vmax = df_plot2['neg_log10_p'].max()

scatter_dummy = ax.scatter([], [], c=[], cmap='Reds', vmin=vmin, vmax=vmax)
module_to_x = {m: i for i, m in enumerate(module_ids_sorted)}

for _, row in df_plot2.iterrows():
    x = module_to_x[row['module']]
    y = pathway_order.index(row['pathway_display'])

    if max_count > min_count:
        size = 300 + 900 * (row['overlap'] - min_count) / (max_count - min_count)
    else:
        size = 600

    if row['significance'] == 'suggestive':
        # Open markers
        ax.scatter(x, y, s=size, c='white',
                   edgecolors='red', linewidth=1.5, alpha=0.85, zorder=3)
    else:
        ax.scatter(x, y, s=size, c=[row['neg_log10_p']],
                   cmap='Reds', vmin=vmin, vmax=vmax,
                   edgecolors='black', linewidth=0.8, alpha=0.85, zorder=2)

ax.set_xticks(range(len(module_ids_sorted)))
ax.set_yticks(range(len(pathway_order)))
ax.set_xticklabels([f'M{m}' for m in module_ids_sorted], rotation=45, ha='right')
ax.set_yticklabels(pathway_order, fontsize=14)
ax.set_xlim(-0.5, len(module_ids_sorted) - 0.5)
ax.set_ylim(-0.5, len(pathway_order) - 0.5)
ax.set_xlabel('Module', fontweight='bold')
ax.set_ylabel('Pathway', fontweight='bold')
ax.set_title(f'Pathway Enrichment ({n_modules_plot} modules, nominal p < 0.05)', fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
for spine in ax.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(1.2)

# Colorbar
cbar = plt.colorbar(scatter_dummy, ax=ax, fraction=0.015, pad=0.02, shrink=0.5)
cbar.set_label(r'$-\log_{10}(p)$', labelpad=10)
cbar.ax.tick_params(labelsize=14)
cbar.outline.set_linewidth(1.2)

# Size legend
legend_counts = [int(min_count), int((min_count + max_count) / 2), int(max_count)]
legend_handles = []
for count in legend_counts:
    if max_count > min_count:
        s = 300 + 900 * (count - min_count) / (max_count - min_count)
    else:
        s = 600
    legend_handles.append(plt.scatter([], [], s=s, c='gray', alpha=0.7, edgecolors='black', linewidth=0.8))

# Suggestive legend
legend_handles.append(plt.scatter([], [], s=400, c='white', edgecolors='red', linewidth=1.5))
legend_labels = [f'{c} genes' for c in legend_counts] + ['suggestive']

legend = ax.legend(
    legend_handles, legend_labels,
    title='Gene Count',
    loc='upper left', bbox_to_anchor=(1.02, 1.0),
    framealpha=0.95, edgecolor='black', fancybox=False,
    labelspacing=3.0, borderpad=1.0
)
legend.get_frame().set_linewidth(1.2)

plt.tight_layout()
plt.savefig(results_dir / 'dtc_module_enrichment_dotplot_nominal.png', dpi=600, bbox_inches='tight')
plt.savefig(results_dir / 'dtc_module_enrichment_dotplot_nominal.svg', format='svg', bbox_inches='tight')
plt.show()

print(f'Saved: {results_dir}/dtc_module_enrichment_dotplot_nominal.png')
print(f'Saved: {results_dir}/dtc_module_enrichment_dotplot_nominal.svg')

# --- CSVSaved ---
df_nom_save = df_nom[['module', 'pathway', 'p_value', 'q_value', 'fold_enrichment', 'overlap', 'pathway_size', 'genes', 'significance']].copy()
df_nom_save['overlap'] = df_nom_save.apply(lambda r: f"{int(r['overlap'])}/{int(r['pathway_size'])}", axis=1)
df_nom_save = df_nom_save.rename(columns={'genes': 'overlap_genes'})
df_nom_save = df_nom_save.drop(columns=['pathway_size'])
df_nom_save = df_nom_save.sort_values(['module', 'p_value'])
df_nom_save.to_csv(results_dir / 'enrichment_nominal_p005.csv', index=False)
print(f'Saved: {results_dir}/enrichment_nominal_p005.csv ({len(df_nom_save)} entries)')

In [ ]:
# Output CSV with functional labels for all modules (unique label assignment)
print('=' * 80)
print('[Functional Labels for All Modules]')
print('=' * 80)

all_modules = sorted(df_enrichment['module'].unique())

# --- Step 1: Create candidate pathway list for each module ---
# Sort criteria: FDR < 0.05 first, then by fold_enrichment descending
#              No FDR -> fold_enrichment descending
module_candidates = {}
for mod in all_modules:
    df_mod = df_enrichment[df_enrichment['module'] == mod].copy()
    df_mod['is_fdr'] = (df_mod['q_value'] < 0.05).astype(int)
    df_mod = df_mod.sort_values(['is_fdr', 'fold_enrichment'], ascending=[False, False])
    module_candidates[mod] = df_mod.reset_index(drop=True)

# --- Step 2: Assign unique labels using greedy algorithm ---
# Priority: More FDR hits -> Higher fold_enrichment of top candidate
module_priority = sorted(all_modules, key=lambda m: (
    -(module_candidates[m]['is_fdr'].sum() if len(module_candidates[m]) > 0 else 0),
    -(module_candidates[m]['fold_enrichment'].iloc[0] if len(module_candidates[m]) > 0 else 0)
))

used_pathways = set()
assigned = {}  # mod -> (row_data, rank_used)

for mod in module_priority:
    df_cands = module_candidates[mod]
    if len(df_cands) == 0:
        assigned[mod] = (None, None)
        continue
    # Find unique pathway (sorted by FDR + fold_enrichment)
    found = False
    for rank, (_, row) in enumerate(df_cands.iterrows()):
        if row['pathway'] not in used_pathways:
            used_pathways.add(row['pathway'])
            assigned[mod] = (row, rank)
            found = True
            break
    if not found:
        # All candidates used -> use top-1 as is（allowing duplicates）
        assigned[mod] = (df_cands.iloc[0], 0)

# --- Step 3: Construct label rows ---
label_rows = []
for mod in all_modules:
    row_data, rank_used = assigned[mod]
    module_size = int((df_modules['module'] == mod).sum())

    if row_data is None:
        label_rows.append({
            'module': mod,
            'module_size': module_size,
            'label_pathway': 'no_test',
            'label_rank': np.nan,
            'p_value': np.nan,
            'q_value': np.nan,
            'fold_enrichment': np.nan,
            'overlap': '0/0',
            'overlap_genes': '',
            'significance': 'none'
        })
        continue

    best = row_data
    if best['q_value'] < 0.05:
        sig = 'FDR'
    elif best['p_value'] < 0.05:
        sig = 'nominal'
    else:
        sig = 'suggestive'

    label_rows.append({
        'module': mod,
        'module_size': module_size,
        'label_pathway': best['pathway'],
        'label_rank': rank_used + 1,  # 1-indexed: 1=top pathway
        'p_value': best['p_value'],
        'q_value': best['q_value'],
        'fold_enrichment': best['fold_enrichment'],
        'overlap': f"{int(best['overlap'])}/{int(best['pathway_size'])}",
        'overlap_genes': best.get('genes', ''),
        'significance': sig
    })

df_labels = pd.DataFrame(label_rows)
df_labels.to_csv(results_dir / 'module_functional_labels.csv', index=False)

# --- Display statistics ---
n_unique = df_labels['label_pathway'].nunique()
n_rank1 = (df_labels['label_rank'] == 1).sum()
n_fallback = (df_labels['label_rank'] > 1).sum()
n_dup = len(df_labels) - n_unique

print(f'Number of modules: {len(df_labels)}')
print(f'  FDR < 0.05: {(df_labels["significance"] == "FDR").sum()}')
print(f'  nominal p < 0.05: {(df_labels["significance"] == "nominal").sum()}')
print(f'  suggestive: {(df_labels["significance"] == "suggestive").sum()}')
print(f'\nUnique labels: {n_unique} / {len(df_labels)}  (Duplicates: {n_dup})')
print(f'  Using top-1 pathway: {n_rank1}')
print(f'  Using alternative pathway: {n_fallback}')
if n_fallback > 0:
    print('  Alternative modules:')
    for _, r in df_labels[df_labels['label_rank'] > 1].iterrows():
        print(f'    M{r["module"]} → rank {int(r["label_rank"])}: {r["label_pathway"]}')
if n_dup > 0:
    dup_pathways = df_labels['label_pathway'][df_labels['label_pathway'].duplicated(keep=False)]
    print(f'  Duplicate pathways:')
    for pw in dup_pathways.unique():
        mods = df_labels[df_labels['label_pathway'] == pw]['module'].tolist()
        print(f'    "{pw}" → M{", M".join(str(m) for m in mods)}')

print(f'\nSaved: {results_dir}/module_functional_labels.csv')
print()
print(df_labels.to_string(index=False))

# CSV for publication (excluding label_rank, p_value, q_value)
df_pub = df_labels.drop(columns=['label_rank', 'module_size'])
df_pub['fold_enrichment'] = df_pub['fold_enrichment'].round(2)
df_pub = df_pub.rename(columns={
    'module': 'Module',
    'label_pathway': 'Pathway',
    'p_value': 'p-value',
    'q_value': 'q-value',
    'fold_enrichment': 'Fold Enrichment',
    'overlap': 'Overlap',
    'overlap_genes': 'Genes',
    'significance': 'Significance',
})
df_pub.to_csv(results_dir / 'module_functional_labels_pub.csv', index=False)
print(f'\nSaved (publication): {results_dir}/module_functional_labels_pub.csv')


In [ ]:
# Compound-module association analysis using Module Eigengene (ME)
from sklearn.decomposition import PCA
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np

print('=' * 80)
print('[Module Eigengene (ME) Calculation]')
print('=' * 80)

# v2: Using df_modules, excluding module 0 (unassigned)
all_modules = sorted([m for m in df_modules['module'].unique() if m != 0])
print(f'Total modules (excluding module 0): {len(all_modules)}')

# Calculate Module Eigengene (1st principal component) for each module
module_eigengenes = {}
me_variance_explained = {}

for module_id in all_modules:
    module_genes = df_modules[df_modules['module'] == module_id]['gene'].tolist()
    
    if len(module_genes) < 2:
        continue
    
    # Module gene expression matrix (compounds x genes)
    available_genes = [g for g in module_genes if g in expression_filtered.columns]
    if len(available_genes) < 2:
        continue
        
    module_expr = expression_filtered[available_genes]
    
    # Standardization
    module_expr_scaled = (module_expr - module_expr.mean()) / module_expr.std()
    module_expr_scaled = module_expr_scaled.dropna(axis=1)
    
    if module_expr_scaled.shape[1] < 2:
        continue
    
    # Run PCA
    pca = PCA(n_components=1)
    me_scores = pca.fit_transform(module_expr_scaled)
    
    # Module Eigengene (score for each compound)
    module_eigengenes[module_id] = pd.Series(me_scores.flatten(), index=expression_filtered.index)
    me_variance_explained[module_id] = pca.explained_variance_ratio_[0]
    
    print(f'M{module_id}: {len(available_genes)}genes, Variance explained = {pca.explained_variance_ratio_[0]:.1%}')

# Convert to DataFrame
df_me = pd.DataFrame(module_eigengenes)
df_me.columns = [f'ME{m}' for m in df_me.columns]

print(f'\nModule Eigengene matrix: {df_me.shape} (compounds x modules)')
print('=' * 80)

# Save: module_eigengenes.csv (compounds x ME scores)
df_me.to_csv(results_dir / 'module_eigengenes.csv')
print(f'Saved: {results_dir / "module_eigengenes.csv"} ({df_me.shape[0]} compounds × {df_me.shape[1]} modules)')

In [ ]:
# Compound Module Eigengene contribution ranking
print('=' * 80)
print('[Compound-Module Contribution (ME score)]')
print('=' * 80)

# Extract top compounds for each module
top_n = 5
compound_rankings = {}

for module_id in module_eigengenes.keys():
    me_col = f'ME{module_id}'
    
    # Get top compounds by absolute value
    top_positive = df_me[me_col].nlargest(top_n)
    top_negative = df_me[me_col].nsmallest(top_n)
    
    compound_rankings[module_id] = {
        'positive': top_positive,
        'negative': top_negative
    }
    
    print(f'\n【M{module_id}】(Variance explained: {me_variance_explained[module_id]:.1%})')
    print(f'  Positive contribution (module activation):')
    for compound, score in top_positive.items():
        print(f'     {compound}: {score:.4f}')
    print(f'  Negative contribution (module suppression):')
    for compound, score in top_negative.items():
        print(f'     {compound}: {score:.4f}')

print('\n' + '=' * 80)

In [ ]:
# Correlation matrix between Module Eigengenes (module specificity evaluation)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print('=' * 80)
print('[Inter-Module Eigengene Correlation Analysis (All Modules)]')
print('=' * 80)

print(f'Total modules: {len(module_eigengenes)}')

me_correlation_all = df_me.corr(method='pearson')

specificity_scores_all = {}
for col in me_correlation_all.columns:
    other_corrs = me_correlation_all[col].drop(col).abs()
    specificity = 1 - other_corrs.mean()
    module_id = int(col.replace('ME', ''))
    specificity_scores_all[col] = {
        'module_id': module_id,
        'specificity': specificity,
        'mean_abs_corr': other_corrs.mean(),
        'max_abs_corr': other_corrs.max(),
        'max_corr_with': other_corrs.idxmax() if len(other_corrs) > 0 else 'N/A',
        'n_genes': len(df_modules[df_modules['module'] == module_id]),
        'var_explained': me_variance_explained.get(module_id, 0)
    }

df_specificity_all = pd.DataFrame(specificity_scores_all).T
df_specificity_sorted = df_specificity_all.sort_values('specificity', ascending=False)

top_n = min(10, len(df_specificity_sorted))
print(f'\n[Module Specificity Score Top{top_n}】')
print('(1 - mean |correlation| with other modules, higher = more specific)')
print('-' * 70)

for rank, (module, row) in enumerate(df_specificity_sorted.head(top_n).iterrows(), 1):
    print(f'  {rank:2}. {module}: Specificity={row["specificity"]:.3f}, Number of genes={int(row["n_genes"])}, Variance explained={row["var_explained"]:.1%}')

top_modules = df_specificity_sorted.head(top_n).index.tolist()
me_correlation_top = me_correlation_all.loc[top_modules, top_modules]

# Figure 1: Correlation heatmap (without dendrogram)
fig1, ax1 = plt.subplots(figsize=(12, 10))
sns.heatmap(
    me_correlation_top,
    cmap='RdBu_r',
    center=0, vmin=-1, vmax=1,
    annot=True, fmt='.2f',
    annot_kws={'size': 16, 'fontweight': 'bold'},
    linewidths=1, linecolor='white',
    square=True,
    cbar_kws={'label': 'Correlation', 'shrink': 0.8},
    ax=ax1
)
ax1.set_xticklabels(ax1.get_xticklabels(), fontweight='bold', rotation=45, ha='right')
ax1.set_yticklabels(ax1.get_yticklabels(), fontweight='bold', rotation=0)
ax1.set_title(f'ME Correlation (Top {top_n} Specific Modules)', fontweight='bold', pad=15)
plt.tight_layout()

output_heatmap = results_dir / 'module_eigengene_correlation_heatmap.png'
plt.savefig(output_heatmap, dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.savefig(output_heatmap.with_suffix('.svg'), format='svg', bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

# Figure 2: Specificity score bar chart
fig2, ax2 = plt.subplots(figsize=(10, 6))
top_specificity = df_specificity_sorted.head(top_n)
colors = ['#d73027' if s > 0.8 else '#fc8d59' if s > 0.6 else '#fee090' if s > 0.4 else '#91bfdb' 
          for s in top_specificity['specificity']]

ax2.barh(range(len(top_specificity)), top_specificity['specificity'], color=colors, edgecolor='black', linewidth=1)
ax2.set_yticks(range(len(top_specificity)))
ax2.set_yticklabels(top_specificity.index, fontweight='bold')
ax2.set_xlabel('Specificity Score (1 - mean|r|)', fontweight='bold')
ax2.set_title(f'Module Specificity Ranking (Top {top_n})', fontweight='bold', pad=10)
ax2.set_xlim(0, 1.1)
ax2.invert_yaxis()

for i, (idx, row) in enumerate(top_specificity.iterrows()):
    ax2.text(row['specificity'] + 0.02, i, f'{row["specificity"]:.3f}', va='center', fontweight='bold')

ax2.grid(axis='x', alpha=0.3, linestyle='--')
ax2.set_axisbelow(True)
plt.tight_layout()

output_ranking = results_dir / 'module_specificity_ranking_top10.png'
plt.savefig(output_ranking, dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.savefig(output_ranking.with_suffix('.svg'), format='svg', bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

print('\nME correlation analysis complete')
print(f'   {output_heatmap.name}')
print(f'   {output_ranking.name}')
print('=' * 80)

In [ ]:
# Gene network within top 10 modules (TOM-based)
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

print('=' * 80)
print('[Intra-Module Gene Network (Top 10 Modules)]')
print('=' * 80)

# Top 10 modules
top_n_modules = min(10, len(df_specificity_sorted))
top_modules_list = df_specificity_sorted.head(top_n_modules).index.tolist()
top_module_ids = [int(m.replace('ME', '')) for m in top_modules_list]

print(f'Target modules: {top_module_ids}')

# TOM threshold
tom_threshold = 0.01

# Create subplots
n_cols = 5
n_rows = (top_n_modules + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(28, 6 * n_rows))
axes = axes.flatten() if n_rows > 1 else [axes] if top_n_modules == 1 else axes

for idx, module_id in enumerate(top_module_ids):
    ax = axes[idx]
    ax.set_facecolor('#f8f8f8')
    
    module_genes = df_modules[df_modules['module'] == module_id]['gene'].tolist()
    n_genes = len(module_genes)
    
    gene_indices = [filtered_genes.index(g) for g in module_genes if g in filtered_genes]
    
    if len(gene_indices) < 2:
        ax.text(0.5, 0.5, f'M{module_id}\n(n={n_genes})\nToo few genes', 
                ha='center', va='center')
        ax.axis('off')
        continue
    
    TOM_module = TOM[np.ix_(gene_indices, gene_indices)]
    module_gene_names = [filtered_genes[i] for i in gene_indices]
    
    G = nx.Graph()
    for gene in module_gene_names:
        G.add_node(gene)
    
    for i in range(len(module_gene_names)):
        for j in range(i + 1, len(module_gene_names)):
            tom_val = TOM_module[i, j]
            if tom_val >= tom_threshold:
                G.add_edge(module_gene_names[i], module_gene_names[j], weight=tom_val)
    
    if G.number_of_edges() > 0:
        pos = nx.spring_layout(G, k=2.0, iterations=50, seed=42)
    else:
        pos = nx.circular_layout(G)
    
    node_degrees = dict(G.degree())
    node_sizes = [max(node_degrees.get(n, 0) * 80 + 300, 300) for n in G.nodes()]
    
    sorted_by_degree = sorted(node_degrees.items(), key=lambda x: x[1], reverse=True)
    top3_genes = [g for g, d in sorted_by_degree[:3] if d > 0]
    
    node_colors = ['#d73027' if n in top3_genes else '#4575b4' for n in G.nodes()]
    
    if G.number_of_edges() > 0:
        edge_weights = [G[u][v]['weight'] * 8 + 0.5 for u, v in G.edges()]
        nx.draw_networkx_edges(G, pos, width=edge_weights, alpha=0.7, edge_color='#666666', ax=ax)
    
    nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors,
                          edgecolors='black', linewidths=1.5, alpha=0.9, ax=ax)
    
    # Display all gene names (top 3 by degree in red, others in black) - gene names in italic
    top3_labels = {g: g for g in top3_genes}
    other_labels = {g: g for g in G.nodes() if g not in top3_genes}
    
    # Top 3 degree gene labels (red)
    if top3_labels:
        nx.draw_networkx_labels(G, pos, labels=top3_labels, font_size=16, 
                               font_weight='bold', font_color='darkred', ax=ax,
                               font_family='sans-serif')
    # Other gene labels (black)
    if other_labels:
        nx.draw_networkx_labels(G, pos, labels=other_labels, font_size=16, 
                               font_weight='bold', font_color='black', ax=ax,
                               font_family='sans-serif')
    
    top3_str = ', '.join(top3_genes[:2]) if top3_genes else 'None'
    ax.set_title(f'M{module_id} (n={n_genes}, edges={G.number_of_edges()})\nTop: {top3_str}', 
                 fontweight='bold')
    ax.axis('off')

# Hide unused axes
for idx in range(top_n_modules, len(axes)):
    axes[idx].axis('off')

plt.suptitle(f'Intra-Module Gene Networks (TOM >= {tom_threshold})', 
             fontweight='bold', y=1.02)
plt.tight_layout()

output_intra_pub = results_dir / 'module_intra_gene_network_publication.png'
plt.savefig(output_intra_pub, dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.savefig(output_intra_pub.with_suffix('.svg'), format='svg', bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()

print(f'\nIntra-module gene network saved')
print(f'   {output_intra_pub.name}')
print('=' * 80)

In [ ]:
# Module-specific compound clustermap (top compounds per module)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

print('=' * 80)
print('[Identification of Module-Specific Compounds]')
print('=' * 80)

# Top 10 specificity modules
top_n_modules = min(10, len(df_specificity_sorted))
top_modules_list = df_specificity_sorted.head(top_n_modules).index.tolist()
print(f'Target modules（SpecificityTop{top_n_modules}）: {top_modules_list}')

df_me_top_modules = df_me[top_modules_list]

# Extract top compounds per module
top_n_per_module = 1
min_effect = 0.5

module_top_compounds_up = {}
module_top_compounds_down = {}

print(f'\nFilter conditions: |ME| >= {min_effect}')
print(f'Top compounds from each module{top_n_per_module}extracted')

for me_col in top_modules_list:
    me_values = df_me_top_modules[me_col]
    positive_vals = me_values[me_values >= min_effect].nlargest(top_n_per_module)
    module_top_compounds_up[me_col] = positive_vals
    negative_vals = me_values[me_values <= -min_effect].nsmallest(top_n_per_module)
    module_top_compounds_down[me_col] = negative_vals

# Display results
print('\n' + '=' * 70)
print('[Module-Specific Compounds (Upregulated/Activated)]')
print('=' * 70)
for me_col in top_modules_list:
    print(f'\n{me_col}:')
    if len(module_top_compounds_up[me_col]) > 0:
        for compound, score in module_top_compounds_up[me_col].items():
            print(f'    ↑ {compound}: ME={score:.3f}')
    else:
        print('    (None)')

print('\n' + '=' * 70)
print('[Module-Specific Compounds (Downregulated/Suppressed)]')
print('=' * 70)
for me_col in top_modules_list:
    print(f'\n{me_col}:')
    if len(module_top_compounds_down[me_col]) > 0:
        for compound, score in module_top_compounds_down[me_col].items():
            print(f'    ↓ {compound}: ME={score:.3f}')
    else:
        print('    (None)')

# Create data for heatmap
all_top_up = set()
all_top_down = set()
for me_col in top_modules_list:
    all_top_up.update(module_top_compounds_up[me_col].index)
    all_top_down.update(module_top_compounds_down[me_col].index)

print(f'\nUpregulated compounds (unique): {len(all_top_up)}entries')
print(f'Downregulated compounds (unique): {len(all_top_down)}entries')

# Data range
all_values = df_me_top_modules.values.flatten()
vmax_abs = np.abs(all_values).max()

# Upregulated (activation) clustermap
if len(all_top_up) > 1:
    df_plot_up = df_me_top_modules.loc[list(all_top_up)]
    
    g1 = sns.clustermap(
        df_plot_up,
        cmap='Reds',
        vmin=0, vmax=vmax_abs,
        annot=True, fmt='.1f',
        annot_kws={'size': 16, 'fontweight': 'bold'},
        linewidths=0.5, linecolor='gray',
        figsize=(14, max(8, len(all_top_up) * 0.6)),
        dendrogram_ratio=(0.12, 0.12),
        
        method='ward',
        row_cluster=True,
        col_cluster=True,
        tree_kws={'linewidths': 1.5}
    )
    g1.ax_heatmap.set_xlabel('Module', fontweight='bold')
    g1.ax_heatmap.set_ylabel('Compound', fontweight='bold')
    g1.ax_heatmap.set_xticklabels(g1.ax_heatmap.get_xticklabels(), fontweight='bold', rotation=45, ha='right')
    g1.ax_heatmap.set_yticklabels(g1.ax_heatmap.get_yticklabels(), rotation=0)
    
    g1.cax.set_ylabel('ME Score', fontweight='bold')
    plt.suptitle(f'Module Activation (Top {top_n_per_module} per Module)', fontweight='bold', y=1.02)
    
    output_up = results_dir / 'module_specific_compounds_activation_clustermap.png'
    plt.savefig(output_up, dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.savefig(output_up.with_suffix('.svg'), format='svg', bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.show()
    print(f'\nUpregulated compound clustermap: {output_up.name}')

# Downregulated (suppression) clustermap
if len(all_top_down) > 1:
    df_plot_down = df_me_top_modules.loc[list(all_top_down)]
    
    g2 = sns.clustermap(
        df_plot_down,
        cmap='Blues_r',
        vmin=-vmax_abs, vmax=0,
        annot=True, fmt='.1f',
        annot_kws={'size': 16, 'fontweight': 'bold'},
        linewidths=0.5, linecolor='gray',
        figsize=(14, max(8, len(all_top_down) * 0.6)),
        dendrogram_ratio=(0.12, 0.12),
        
        method='ward',
        row_cluster=True,
        col_cluster=True,
        tree_kws={'linewidths': 1.5}
    )
    g2.ax_heatmap.set_xlabel('Module', fontweight='bold')
    g2.ax_heatmap.set_ylabel('Compound', fontweight='bold')
    g2.ax_heatmap.set_xticklabels(g2.ax_heatmap.get_xticklabels(), fontweight='bold', rotation=45, ha='right')
    g2.ax_heatmap.set_yticklabels(g2.ax_heatmap.get_yticklabels(), rotation=0)
    
    g2.cax.set_ylabel('ME Score', fontweight='bold')
    plt.suptitle(f'Module Suppression (Top {top_n_per_module} per Module)', fontweight='bold', y=1.02)
    
    output_down = results_dir / 'module_specific_compounds_suppression_clustermap.png'
    plt.savefig(output_down, dpi=600, bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.savefig(output_down.with_suffix('.svg'), format='svg', bbox_inches='tight', facecolor='white', edgecolor='none')
    plt.show()
    print(f'Downregulated compound clustermap: {output_down.name}')

print('=' * 80)